# Algebraic Simplification of Regular Expressions

In Section 2.3 of the lecture notes, we have seen a number of algebraic laws for regular expressions.  For
example, the law
$$ \varepsilon \cdot r \doteq r $$
tells us that the regular expression $\varepsilon \cdot r$ describes the same language as the regular
expression $r$.  If we read such a law from left to right, we can use it to *simplify* a regular expression.

This notebook implements a program that simplifies regular expressions in this way.  In addition, it contains
an *interpreter* for regular expressions.  The notebook consists of five parts:
1. A *scanner* splits a string like `'(ε + a)*·b'` into a list of tokens.
2. A hand-written *recursive descent parser* turns this list of tokens into an *abstract syntax tree*.

   For the moment, we will not discuss the *scanner* and the *parser*, as these are topics that will only be covered later in this lecture.
3. The function `to_str` turns an abstract syntax tree back into a string.
4. The function `matches` is an interpreter: it checks whether a string is an element of the language that is
   described by a regular expression.
5. The function `simplify_regexp` simplifies an abstract syntax tree using the laws of Section 2.3.

## The Syntax of Regular Expressions

We use the syntax of Chapter 2:
- `∅` denotes the empty language and `ε` denotes the language that contains only the empty string.
- Every letter and every digit is a character of the alphabet $\Sigma$.
- `+` denotes the union of two languages, `·` denotes their product, and the postfix operator `*` denotes
  the Kleene closure.
- Parentheses can be used for grouping.

The postfix operator `*` has the highest precedence, followed by `·`, while `+` has the lowest precedence.
Both `+` and `·` associate to the left.  Blanks are ignored.

**Note:** In *Jupyter*, the symbols `∅` and `ε` can be entered by typing `\emptyset` and `\varepsilon`,
followed by the tabulator key.  Typing `\cdot` followed by the tabulator key yields the symbol `⋅`, which
looks almost like `·` but is a different Unicode character.  Therefore, our scanner accepts both of them.

## The Scanner

Every token of a regular expression consists of a single character.  Therefore, we do not need regular
expressions to implement our scanner: the function `tokenize(s)` looks at the characters of the string `s`
one by one.
- Blanks and other white space characters are skipped.
- The character `⋅` is replaced by `·`, so that the parser only has to deal with `·`.
- The method `c.isalnum()` checks whether `c` is a letter or a digit.  Note that `ε` is a Greek letter.
  Nevertheless, it is not a character of our alphabet, since it is listed among the special symbols, too.
  The parser checks for `ε` before it checks for characters.
- Every other character is an error.

In [ ]:
def tokenize(s):
    tokens = []
    for c in s:
        if c.isspace():
            continue
        if c == '⋅':
            c = '·'
        if c in '∅ε+·*()' or c.isalnum():
            tokens.append(c)
        else:
            raise SyntaxError(f'Illegal character {c!r} in {s!r}.')
    return tokens

Let's test the scanner.

In [ ]:
tokenize('(ε + a)* ⋅ b')

## Abstract Syntax Trees

We represent regular expressions as *nested tuples* that are built from strings and tuples.  The nested tuples 
are known as *abstract syntax trees*:
- The regular expressions `∅` and `ε` are represented by the strings `'∅'` and `'ε'`.
- A character `c` from the alphabet is represented by the string `c`.
- The regular expression $r_1 + r_2$ is represented by the tuple `('+', t1, t2)`, where `t1` and `t2` are
  the representations of $r_1$ and $r_2$.
- The regular expression $r_1 \cdot r_2$ is represented by the tuple `('·', t1, t2)`.
- The regular expression $r^*$ is represented by the tuple `('*', t)`, where `t` is the representation of $r$.

Parentheses do not show up in the abstract syntax tree, since $L\bigl((r)\bigr) = L(r)$.  They are only
needed to determine the structure of the tree.  For example, `a + b·c*` is represented by
```
('+', 'a', ('·', 'b', ('*', 'c')))
```

## The Parser

Our parser is a *recursive descent parser* that is based on the following grammar:
$$
  \begin{array}{lcl}
  \mathrm{regexp}  & \rightarrow & \mathrm{product}\;\;\bigl(\texttt{'+'}\;\; \mathrm{product}\bigr)^* \\[0.2cm]
  \mathrm{product} & \rightarrow & \mathrm{factor} \;\;\bigl(\texttt{'·'}\;\; \mathrm{factor}\bigr)^*  \\[0.2cm]
  \mathrm{factor}  & \rightarrow & \mathrm{atom} \;\;\texttt{'*'}^*                                    \\[0.2cm]
  \mathrm{atom}    & \rightarrow & \texttt{'('} \;\;\mathrm{regexp} \;\;\texttt{')'}
                     \;\mid\; \texttt{'∅'} \;\mid\; \texttt{'ε'} \;\mid\; \texttt{CHAR}
  \end{array}
$$
Here, the operator ${}^*$ in the grammar means that the part before it can be repeated any number of times.
For every grammar symbol there is a function that parses this grammar symbol.  Every one of these functions
takes a list of tokens `TL` as its argument and returns a pair `(tree, Rest)`, where
- `tree` is the representation of the part of `TL` that has been parsed, and
- `Rest` is the list of those tokens that have not yet been consumed.

The function `parse_regexp(TL)` implements the rule for $\mathrm{regexp}$:
- First, it parses a product.
- As long as the next token is `+`, it parses another product and combines the tree built so far with the
  new tree.  Since the new tree becomes the right argument of `+`, the operator `+` associates to the left.

In [ ]:
def parse_regexp(TL):
    result, Rest = parse_product(TL)
    while Rest != [] and Rest[0] == '+':
        arg, Rest = parse_product(Rest[1:])
        result    = ('+', result, arg)
    return result, Rest

The function `parse_product(TL)` implements the rule for $\mathrm{product}$ in the same way.

In [ ]:
def parse_product(TL):
    result, Rest = parse_factor(TL)
    while Rest != [] and Rest[0] == '·':
        arg, Rest = parse_factor(Rest[1:])
        result    = ('·', result, arg)
    return result, Rest

The function `parse_factor(TL)` implements the rule for $\mathrm{factor}$.  It first parses an atom.  Then,
every `*` that follows wraps the tree built so far into a tuple of the form `('*', tree)`.  Hence, `a**` is
represented as `('*', ('*', 'a'))`.

In [ ]:
def parse_factor(TL):
    result, Rest = parse_atom(TL)
    while Rest != [] and Rest[0] == '*':
        result = ('*', result)
        Rest   = Rest[1:]
    return result, Rest

The function `parse_atom(TL)` implements the rule for $\mathrm{atom}$.
- If the first token is an opening parenthesis, we parse a regular expression, which has to be followed by a
  closing parenthesis.
- The tokens `∅` and `ε` and all characters are their own abstract syntax trees.  The test for `∅` and `ε`
  has to come first, since `ε` is a letter, too.
- Every other token, e.g. `+` or `)`, cannot start an atom and hence is an error.  If the list `TL` is
  empty, the input has ended too early.

In [ ]:
def parse_atom(TL):
    if TL == []:
        raise SyntaxError('Unexpected end of input.')
    if TL[0] == '(':
        result, Rest = parse_regexp(TL[1:])
        if Rest == [] or Rest[0] != ')':
            raise SyntaxError(f"')' expected, but found {Rest[:1]}.")
        return result, Rest[1:]
    if TL[0] in '∅ε' or TL[0].isalnum():
        return TL[0], TL[1:]
    raise SyntaxError(f'Unexpected token {TL[0]!r}.')

The function `parse(s)` puts the pieces together: it tokenizes the string `s`, parses the resulting list of
tokens, and checks that all tokens have been consumed.

In [ ]:
def parse(s):
    tree, Rest = parse_regexp(tokenize(s))
    if Rest != []:
        raise SyntaxError(f'Unexpected token {Rest[0]!r}.')
    return tree

In [ ]:
parse('a + b·c*')

Let us check that `+` and `·` associate to the left.

In [ ]:
parse('a + b + c')

In [ ]:
parse('a·b·c')

## Turning Trees back into Strings

The function `to_str(r)` turns the abstract syntax tree `r` into a string.  It only uses parentheses where
they are needed.  To decide this, the function `precedence(r)` returns the precedence of the operator at the
root of `r`: atoms bind tighter than every operator, so they get the highest precedence.

In [ ]:
def precedence(r):
    if isinstance(r, str):
        return 4
    return { '*': 3, '·': 2, '+': 1 }[r[0]]

The function `to_str` uses the helper function `arg_str(r, p)`, which puts parentheses around the string of
`r` if the precedence of `r` is lower than `p`.
- The argument of `*` needs parentheses unless it is an atom or another `*` expression.
- The left argument of a binary operator needs parentheses if its precedence is lower than the precedence of
  the operator.
- Since `+` and `·` associate to the left, the right argument needs parentheses already if its precedence is
  the same as the precedence of the operator.  For example, the tree `('+', 'a', ('+', 'b', 'c'))` is printed
  as `a + (b + c)`.

In [ ]:
def arg_str(r, p):
    if precedence(r) < p:
        return '(' + to_str(r) + ')'
    return to_str(r)

def to_str(r):
    if isinstance(r, str):
        return r
    op = r[0]
    if op == '*':
        return arg_str(r[1], 3) + '*'
    p   = precedence(r)
    sep = ' + ' if op == '+' else '·'
    return arg_str(r[1], p) + sep + arg_str(r[2], p + 1)

In [ ]:
to_str(parse('(a + b + c)·c*·(d·(e·f))*'))

In [ ]:
to_str(('+', 'a', ('+', 'b', 'c')))

## An Interpreter for Regular Expressions

In Chapter 2, the meaning of a regular expression $r$ is defined as the language $L(r)$.  The function
`matches(r, s)` checks whether the string `s` is an element of the language $L(r)$ of the regular expression
`r`, which is given as a string.  It parses `r` and then calls the function `member(t, s)`, which does the
real work on the abstract syntax tree `t`.

The function `member(t, s)` follows the inductive definition of $L(t)$ in Section 2.2 of the lecture notes:
- $L(\emptyset) = \{\}$: no string is an element of the empty language.
- $L(\varepsilon) = \{\lambda\}$: only the empty string is an element.
- $L(c) = \{c\}$: only the string consisting of the character $c$ is an element.
- $L(r_1 + r_2) = L(r_1) \cup L(r_2)$: the string `s` has to be an element of $L(r_1)$ or of $L(r_2)$.
- $L(r_1 \cdot r_2) = L(r_1) \cdot L(r_2)$: the string `s` has to be split into two parts `s[:k]` and `s[k:]`
  such that `s[:k]` is an element of $L(r_1)$ and `s[k:]` is an element of $L(r_2)$.  We try every position
  `k` from $0$ up to `len(s)`.
- $L(r^*) = \bigcup_{n \in \mathbb{N}} L(r)^n$: the string `s` is an element if it is empty, since
  $\lambda \in L(r)^0$.  Otherwise, `s` has to be split into a first part `s[:k]` that is an element of
  $L(r)$ and a rest `s[k:]` that is an element of $L(r^*)$.  Here, we only try positions `k` from $1$ on:
  if the first part were empty, the rest would be the string `s` itself, and the recursive call would never
  terminate.  We do not lose anything this way, since empty parts do not contribute to the string `s`.

Trying all positions `k` can lead to the same question being asked many times.  For example, when matching
`(a + a·a)*` against a long string of `a`s, the same suffix of `s` is checked over and over again.  Therefore,
we use the *decorator* `@cache` from the module `functools`.  It stores the result of every call of `member`
and returns the stored result when the function is called again with the same arguments.  This works since
abstract syntax trees are tuples and strings, which can be used as keys of a dictionary.

In [ ]:
from functools import cache

In [ ]:
@cache
def member(t, s):
    if t == '∅':
        return False
    if t == 'ε':
        return s == ''
    if isinstance(t, str):
        return s == t
    if t[0] == '+':
        return member(t[1], s) or member(t[2], s)
    if t[0] == '·':
        return any(member(t[1], s[:k]) and member(t[2], s[k:]) for k in range(len(s) + 1))
    # t[0] == '*'
    if s == '':
        return True
    return any(member(t[1], s[:k]) and member(t, s[k:]) for k in range(1, len(s) + 1))

In [ ]:
def matches(r, s):
    return member(parse(r), s)

### Testing the Interpreter

The list `tests` contains triples of the form `(r, s, expected)`, where `expected` is the result that
`matches(r, s)` should return.  Most of the regular expressions are taken from Chapters 1 and 2 of the
lecture notes:
- $(\mathtt{a} + \mathtt{b} + \mathtt{c}) \cdot (\mathtt{a} + \mathtt{b} + \mathtt{c})$ describes all
  strings of length $2$.
- $(\mathtt{a} + \mathtt{b} + \mathtt{c}) \cdot (\mathtt{a} + \mathtt{b} + \mathtt{c})^*$ describes all
  strings of length at least $1$.
- $(\mathtt{b} + \mathtt{c})^* \cdot \mathtt{a} \cdot (\mathtt{b} + \mathtt{c})^*$ describes all strings that
  contain exactly one `a`.
- $\mathtt{1} \cdot (\mathtt{0} + \mathtt{1})^* + \mathtt{0}$ describes the language $L_\mathbb{N}$ of
  Chapter 1, i.e. the natural numbers in binary notation.

In [ ]:
tests = [ ('∅',                                    '',        False),
          ('ε',                                    '',        True ),
          ('ε',                                    'a',       False),
          ('a',                                    'a',       True ),
          ('a',                                    'aa',      False),
          ('∅*',                                   '',        True ),
          ('(a + b + c)·(a + b + c)',              'ab',      True ),
          ('(a + b + c)·(a + b + c)',              'abc',     False),
          ('(a + b + c)·(a + b + c)*',             '',        False),
          ('(a + b + c)·(a + b + c)*',             'cab',     True ),
          ('(b + c)*·a·(b + c)*',                  'bcacb',   True ),
          ('(b + c)*·a·(b + c)*',                  'bcb',     False),
          ('(b + c)*·a·(b + c)*',                  'abca',    False),
          ('1·(0 + 1)* + 0',                       '100',     True ),
          ('1·(0 + 1)* + 0',                       '010',     False),
          ('1·(0 + 1)* + 0',                       '0',       True ),
          ('(a·b)*',                               'ababab',  True ),
          ('(a·b)*',                               'aba',     False),
          ('(ε + a)*',                             'aaa',     True ),
          ('(a + a·a)*·b',                         'a' * 50 + 'b', True ),
          ('(a + a·a)*·b',                         'a' * 50,  False),
        ]

The function `run_tests` checks every triple and prints the result.  If a result differs from the expected
one, the line is marked with `ERROR`.  For long strings, only the first $20$ characters are shown.

In [ ]:
def run_tests(tests):
    for r, s, expected in tests:
        result = matches(r, s)
        mark   = '' if result == expected else '   ERROR'
        shown  = s if len(s) <= 20 else s[:20] + '...'
        print(f'{r:25} {shown!r:26} {result}{mark}')

In [ ]:
run_tests(tests)

## Sums and Products as Lists

Because of the laws

(b) $\;(r_1 + r_2) + r_3 \doteq r_1 + (r_2 + r_3)$ and

(c) $\;(r_1 \cdot r_2) \cdot r_3 \doteq r_1 \cdot (r_2 \cdot r_3)$,

it does not matter how the parentheses are set in a sum or a product of several regular expressions.
Therefore, it is convenient to view a sum as a list of *alternatives* and a product as a list of *factors*.
- The function `summands(r)` returns the list of all regular expressions that are combined with `+` at the top
  of `r`.  If the top operator of `r` is not `+`, the list only contains `r` itself.
- The function `factors(r)` does the same for `·`.
- The function `make_sum(L)` builds a sum from a list of regular expressions.  The sum of an empty list is
  `∅`, since $L(\emptyset) = \{\}$ is the neutral element of the union.
- The function `make_product(L)` builds a product.  The product of an empty list is `ε`, since
  $L(\varepsilon) = \{\lambda\}$ is the neutral element of the product of languages.

Both `make_sum` and `make_product` build trees that associate to the left, just like the parser does.

In [ ]:
def summands(r):
    if isinstance(r, tuple) and r[0] == '+':
        return summands(r[1]) + summands(r[2])
    return [r]

def factors(r):
    if isinstance(r, tuple) and r[0] == '·':
        return factors(r[1]) + factors(r[2])
    return [r]

def make_sum(L):
    if L == []:
        return '∅'
    result = L[0]
    for r in L[1:]:
        result = ('+', result, r)
    return result

def make_product(L):
    if L == []:
        return 'ε'
    result = L[0]
    for r in L[1:]:
        result = ('·', result, r)
    return result

In [ ]:
summands(parse('a + (b + c)·d + e'))

In [ ]:
factors(parse('a·(b·c)·d*'))

Finally, the function `size(r)` counts the nodes of the tree `r`.  We use it to decide whether a
transformation really makes a regular expression simpler.

In [ ]:
def size(r):
    if isinstance(r, str):
        return 1
    return 1 + sum(size(arg) for arg in r[1:])

## Simplifying Products

The function `simplify_product(fs)` simplifies a product, which is given as the list `fs` of its factors.
These factors have already been simplified.
- By law (d), $\emptyset \cdot r \doteq r \cdot \emptyset \doteq \emptyset$, a product that contains `∅`
  as a factor is equivalent to `∅`.
- By law (e), $\varepsilon \cdot r \doteq r \cdot \varepsilon \doteq r$, all factors that are equal to `ε`
  can be dropped.  If no factor remains, `make_product` returns `ε`.

In [ ]:
def simplify_product(fs):
    if '∅' in fs:
        return '∅'
    return make_product([f for f in fs if f != 'ε'])

## Simplifying the Kleene Closure

The function `simplify_star(s)` simplifies the regular expression $s^*$.  The argument `s` has already been
simplified.
- By the laws (k), $\emptyset^* \doteq \varepsilon$, and (l), $\varepsilon^* \doteq \varepsilon$, the
  closure of `∅` and of `ε` is `ε`.
- By law (j), $(r^*)^* \doteq r^*$, the closure of a closure is the closure itself.
- By law (n), $(\varepsilon + r)^* \doteq r^*$, the summand `ε` can be dropped from a sum that occurs as the
  argument of `*`.  Since the sum might then consist of a single summand, which might be a closure itself,
  we call `simplify_star` again.

In [ ]:
def simplify_star(s):
    if s in ('∅', 'ε'):
        return 'ε'
    if isinstance(s, tuple) and s[0] == '*':
        return s
    alternatives = summands(s)
    if 'ε' in alternatives:
        return simplify_star(make_sum([a for a in alternatives if a != 'ε']))
    return ('*', s)

## Simplifying Sums

Sums need the most work.  The function `simplify_sum(alts)` simplifies a sum, which is given as the list
`alts` of its summands.  These summands have already been simplified.
- By law (f), $\emptyset + r \doteq r + \emptyset \doteq r$, all summands that are equal to `∅` are dropped.
- By law (i), $r + r \doteq r$, a summand that occurs twice is dropped.  Because of law (a),
  $r_1 + r_2 \doteq r_2 + r_1$, this works even when the two occurrences are not next to each other.
- If `ε` is one of the summands, the function `absorb_epsilon` tries to get rid of it.  This is explained
  below.
- Finally, the function `factor_out` uses the laws of distributivity to factor out common parts of the
  summands.  This is explained below, too.

In [ ]:
def simplify_sum(alts):
    unique = []
    for a in alts:
        if a != '∅' and a not in unique:
            unique.append(a)
    if 'ε' in unique:
        unique = absorb_epsilon(unique)
    return factor_out(unique)

### Absorbing `ε`

Law (m) states that $\varepsilon + r^* \cdot r \doteq r^*$, i.e. the sum $\varepsilon + r^* \cdot r$ can be
replaced by $r^*$.  Because of the laws (a) and (b), the summands
$\varepsilon$ and $r^* \cdot r$ do not have to be next to each other.

In addition, the summand $\varepsilon$ can be dropped if another summand is a closure $r^*$, since
$\varepsilon \in L(r^*)$.  This is not one of the laws of Section 2.3, but it follows from them:
$$
  \varepsilon + r^* \;\stackrel{(m)}{\doteq}\; \varepsilon + (\varepsilon + r^* \cdot r)
               \;\stackrel{(b)}{\doteq}\; (\varepsilon + \varepsilon) + r^* \cdot r
               \;\stackrel{(i)}{\doteq}\; \varepsilon + r^* \cdot r
               \;\stackrel{(m)}{\doteq}\; r^*.
$$

The function `absorb_epsilon(alts)` gets a list of summands that contains `ε`.
- The function `is_star_times_base(a)` checks whether `a` has the form $r^* \cdot r$.  If `a` is a product
  with the factors $f_1, \cdots, f_n$, this is the case if $f_1$ has the form $r^*$ and the factors of $r$
  are $f_2, \cdots, f_n$.  For example, `(a·b)*·a·b` has this form.
- If one of the summands has the form $r^* \cdot r$, it is replaced by $r^*$ and `ε` is dropped.
- If one of the summands is a closure, `ε` is dropped.

In [ ]:
def is_star_times_base(a):
    fs = factors(a)
    return (len(fs) >= 2 and isinstance(fs[0], tuple) and fs[0][0] == '*'
            and factors(fs[0][1]) == fs[1:])

def absorb_epsilon(alts):
    for i, a in enumerate(alts):
        if is_star_times_base(a):
            alts = alts[:i] + [factors(a)[0]] + alts[i+1:]
            return [b for b in alts if b != 'ε']
    for a in alts:
        if isinstance(a, tuple) and a[0] == '*':
            return [b for b in alts if b != 'ε']
    return alts

### Factoring out Common Parts

The laws (g) and (h) are the laws of distributivity:

(g) $\;(r_1 + r_2) \cdot r_3 \doteq r_1 \cdot r_3 + r_2 \cdot r_3$,

(h) $\;r_1 \cdot (r_2 + r_3) \doteq r_1 \cdot r_2 + r_1 \cdot r_3$.

Read from right to left, they allow us to factor out a common *suffix* or a common *prefix* of two summands.
For example, $\mathtt{a} \cdot \mathtt{b} + \mathtt{a} \cdot \mathtt{c}$ can be simplified to
$\mathtt{a} \cdot (\mathtt{b} + \mathtt{c})$.

The function `common_prefix(fs1, fs2)` returns the length of the longest common prefix of the lists `fs1` and
`fs2`.

In [ ]:
def common_prefix(fs1, fs2):
    k = 0
    while k < len(fs1) and k < len(fs2) and fs1[k] == fs2[k]:
        k += 1
    return k

The function `factor_candidates(alts)` generates all sums that result from factoring out a common prefix or
a common suffix of two summands.
- For every pair of summands `alts[i]` and `alts[j]`, we compute the lists `fi` and `fj` of their factors.
- If these lists have a common prefix of length `k > 0`, we build the product of this prefix with the sum of
  the remaining factors.  For example, for the summands `a·b` and `a·c`, we build `a·(b + c)`.  If all
  factors of a summand belong to the prefix, the remaining factors form the product `ε`: `a·b + a` becomes
  `a·(b + ε)`.
- To handle a common suffix, we reverse the lists of factors, compute the common prefix of the reversed
  lists, and reverse the results again.
- The new summand replaces `alts[i]`, while `alts[j]` is removed.

In [ ]:
def factor_candidates(alts):
    for i in range(len(alts)):
        for j in range(i + 1, len(alts)):
            fi, fj = factors(alts[i]), factors(alts[j])
            rest   = alts[:i] + alts[i+1:j] + alts[j+1:]
            k = common_prefix(fi, fj)
            if k > 0:
                inner = ('+', make_product(fi[k:]), make_product(fj[k:]))
                yield [make_product(fi[:k] + [inner])] + rest
            k = common_prefix(fi[::-1], fj[::-1])
            if k > 0:
                inner = ('+', make_product(fi[:-k]), make_product(fj[:-k]))
                yield [make_product([inner] + fi[-k:])] + rest

Factoring out does not always make a regular expression simpler: `a·b + a` has the same size as
`a·(b + ε)`.  However, the new sum `b + ε` might be simplified further.  For example, `a·b* + a` becomes
`a·(b* + ε)` and then `a·b*`.  Therefore, the function `factor_out(alts)` simplifies every candidate and only
accepts it if it is smaller than the original sum.  The first candidate that is smaller wins.  If there is no
such candidate, the sum is returned unchanged.

Since every candidate has fewer summands than the original sum, the recursive calls terminate.  However,
since all pairs of summands are tried, the running time grows quickly with the number of summands.  For the
small regular expressions we are interested in, this is no problem.

In [ ]:
def factor_out(alts):
    current = make_sum(alts)
    for candidate in factor_candidates(alts):
        simplified = simplify(make_sum(candidate))
        if size(simplified) < size(current):
            return simplified
    return current

## Putting it all Together

The function `simplify(r)` simplifies the abstract syntax tree `r` *bottom up*: it first simplifies the
arguments of the operator at the root of `r`, and then it simplifies the root itself.
- Atoms cannot be simplified.
- For a closure, the argument is simplified and then `simplify_star` is called.
- For a sum, we collect all summands, simplify them, and call `simplify_sum`.  If a simplified summand is a
  sum itself, it is split into its summands, so that law (b) can be applied.
- Products are handled in the same way as sums.

In [ ]:
def simplify(r):
    if isinstance(r, str):
        return r
    if r[0] == '*':
        return simplify_star(simplify(r[1]))
    if r[0] == '+':
        return simplify_sum([s for a in summands(r) for s in summands(simplify(a))])
    return simplify_product([f for a in factors(r) for f in factors(simplify(a))])

Simplifying a regular expression might create new opportunities for simplification.  For example, when an
inner sum shrinks to a single summand, the product around it might be simplified further.  Therefore, the
function `simplify_regexp(s)` calls `simplify` until the result no longer changes.  It takes a string `s`
and returns the simplified regular expression as a string.

In [ ]:
def simplify_regexp(s):
    r = parse(s)
    while True:
        simplified = simplify(r)
        if simplified == r:
            return to_str(r)
        r = simplified

## Trying it Out

The function `test(s)` simplifies the regular expression `s` and prints both the original and the simplified
regular expression.

In [ ]:
def test(s):
    print(f'{s:42} ⟶  {simplify_regexp(s)}')

We start with the regular expressions from the section *Check your Understanding* of Chapter 2, where we
use `a` for $r$.

In [ ]:
test('∅*')
test('ε*')
test('ε + a*·a')
test('(ε + a)*')

The following regular expressions are more complicated.  In most of them, several laws have to be applied
one after the other.  For example, in the first regular expression
- `a·∅` is simplified to `∅` by law (d), and then `ε + ∅` is simplified to `ε` by law (f),
- `b + b + ∅` is simplified to `b` by the laws (f) and (i),
- `c·ε + ∅` is simplified to `c` by the laws (e) and (f), and
- finally, the factor `ε` is dropped by law (e).

In [ ]:
for s in ['(ε + a·∅)·(b + b + ∅)*·(c·ε + ∅)',
          'a·b·c + a·b·d + a·e',
          'ε + (a + b)*·(a + b) + ∅*',
          '(a·b + a·c)·d + (a·b + a·c)·e',
          '((ε + a*)·(b·ε + ∅))* + ε',
          'x·y*·z + x·z',
          '(ε + (a·b)*·a·b)·c + ∅*·(ε + ∅)·c',
          '(((a + ∅)* + ε)*·b + a*·b)·(ε + c + ∅)',
          '(a + b·∅)*·(a + b·∅) + ((ε))',
          '((x + y)·z + (x + y)·w)* + ε·(∅ + ε)'
         ]:
    test(s)

Our simplifier does not find every possible simplification.  For example, the regular expressions
`ε + a·a*` and `a*` are equivalent, but none of the laws of Section 2.3 can be applied to `ε + a·a*`: law
(m) requires the closure to be the *first* factor.  In Chapter 5, we will see an algorithm that decides
whether two regular expressions are equivalent.

In [ ]:
test('ε + a·a*')

Finally, the parser reports syntax errors.

In [ ]:
for s in ['a + ', '(a·b', 'a # b']:
    try:
        parse(s)
    except SyntaxError as error:
        print(f'{s!r:10}: {error}')